# 3.1.1, Proof of the Cartan relations on 0-forms and 1-forms (operator-equation method)

**Claim.** All six Cartan relations below close when applied to the
generator pair of $\Omega^\bullet(M)$, a 0-form $f$ and a 1-form $df$:

$$
\begin{aligned}
(1)\quad & d^2 = 0 \\
(2)\quad & d\,\mathcal{L}_X - \mathcal{L}_X\,d = 0 \\
(3)\quad & d\,\iota_X + \iota_X\,d = \mathcal{L}_X \\
(4)\quad & \mathcal{L}_X\mathcal{L}_Y - \mathcal{L}_Y\mathcal{L}_X = \mathcal{L}_{[X,Y]} \\
(5)\quad & \mathcal{L}_X\iota_Y - \iota_Y\mathcal{L}_X = \iota_{[X,Y]} \\
(6)\quad & \iota_X\iota_Y + \iota_Y\iota_X = 0
\end{aligned}
$$

**Strategy.** Each identity is expressed as an *operator equation*. The
`AgreementOnGenerators` strategy: if two graded derivations agree on
the generator set of $\Omega^\bullet(M)$, they agree on the whole
algebra (which is closed under wedge and $d$). The generator set is
$\{f, df\}$, i.e. **a 0-form $f$ and a 1-form $df$**. Proof: show
that for each generator both sides reduce to the same normal form via
`ExpandAndSimplify`.

In [1]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.algebra.derivation import Act, Derivation, compose
from jacopy.brackets.lie import LieBracket
from jacopy.calculus.cartan import CartanCalculus, RELATIONS
from jacopy.calculus.exterior_algebra import ExteriorAlgebra
from jacopy.calculus.exterior_d import d
from jacopy.calculus.interior import interior
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import default_engine
from jacopy.proof.verifier import prove_operator_equation

## 1. Setup, algebra, vector fields, Cartan package

* `f`: 0-form generator → `Graded(degree=0)`.
* Generators of `ExteriorAlgebra((f,))`: `(f, df)`, i.e. the
  **0-form + 1-form** pair.
* `X, Y`: degree-0 derivations (vector fields).
* Cartan-mode $\mathcal{L}_X$: defined inside the package as
  $d\circ\iota_X + \iota_X\circ d$ (relation (3) holds *by
  definition*).
* Vector-field bracket: `LieBracket()`, composition-based
  $[X, Y] = X\cdot Y - Y\cdot X$.

In [2]:
reg = PropertyRegistry()

f = Symbol("f")
reg.declare(f, Graded(degree=0))
algebra = ExteriorAlgebra((f,))

X = Derivation("X", degree=0)
Y = Derivation("Y", degree=0)

lie = lambda V: lie_derivative(V, definition="cartan")
iota = interior

cartan = CartanCalculus(
    d=d,
    lie_derivative=lie,
    interior=iota,
    vector_bracket=LieBracket(),
)

engine = default_engine(registry=reg)

print("Generators (algebra.generators):")
for g in algebra.generators:
    prop = reg.get(g, Graded)
    deg = prop.degree if prop is not None else "(generated)"
    print(f"  {g}   (degree {deg})")
print(f"\nVector fields  : X (deg 0), Y (deg 0)")
print(f"Lie-mode       : cartan (i.e. L_X := d∘ι_X + ι_X∘d)")
print(f"VF bracket     : {cartan.vector_bracket.name}")


Generators (algebra.generators):
  f   (degree 0)
  d(f)   (degree (generated))

Vector fields  : X (deg 0), Y (deg 0)
Lie-mode       : cartan (i.e. L_X := d∘ι_X + ι_X∘d)
VF bracket     : [·,·]


## 2. Visualization helper

A small helper for showing the equality between two sides on each
generator: apply an operator to a generator and reduce to normal form
via engine + simplify.

In [3]:
from jacopy.algorithms.simplify import simplify
from jacopy.algorithms.product_rule import product_rule

def normal_form(op_expr, generator):
    """Apply an operator expression to a generator and run the engine to a fixed point."""
    current = Act(op_expr, generator)
    for _ in range(40):
        expanded, _ = engine.expand(current)
        after = simplify(product_rule(expanded, reg), reg)
        if after == current:
            return current
        current = after
    return current


def show_relation(name, lhs_op, rhs_op):
    """Open the relation on each generator and display the equality."""
    print(f"--- {name} ---")
    for g in algebra.generators:
        lhs_val = normal_form(lhs_op, g)
        rhs_val = normal_form(rhs_op, g)
        eq = "OK" if lhs_val == rhs_val else "FAIL"
        print(f"  {eq}  generator {g!s:<6}  LHS({g}) = {lhs_val}")
        print(f"        {' '*len(str(g)):<6}  RHS({g}) = {rhs_val}")
    print()


def show_branches(chain, *, label=""):
    """Print the per-generator branches of an AgreementOnGenerators chain."""
    parent = chain.steps[0]
    if label:
        print(f"### {label}, sub-steps (AgreementOnGenerators branches)")
    for branch in parent.children:
        print(f"\n  +-- generator branch: {branch.before} -> {branch.after}")
        print(f"  |   ({len(branch.children)} sub-step(s))")
        for i, sub in enumerate(branch.children):
            print(f"  |  [{i}] {sub.rule}")
            print(f"  |       {sub.before}  ->  {sub.after}")
        print(f"  +--")


## 3. Relation (1), $d^2 = 0$

**Operator equation:** $d \circ d \equiv 0$ (on every form).

**Per generator:**
* $d^2(f) = d(df) \xrightarrow{d^2 = 0\text{ axiom}} 0$
* $d^2(df) = d(d(df)) \xrightarrow{d^2 = 0\text{ axiom}} 0$

In the engine, the `DSquaredZeroDefinition` axiom directly reduces the
shape `Act(d, Act(d, x))` to $0$.

In [4]:
chain1 = cartan.verify("d_squared_zero", algebra=algebra, registry=reg)
print(f"verify(d_squared_zero)  ->  CLOSED ({len(chain1)} steps, root rule: {chain1.steps[0].rule!r})")
print()
show_relation("(1) d^2 = 0", lhs_op=compose(d, d), rhs_op=Integer(0))
show_branches(chain1, label="(1) d^2 = 0")


verify(d_squared_zero)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (1) d^2 = 0 ---
  OK  generator f       LHS(f) = 0
                RHS(f) = 0
  OK  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (1) d^2 = 0, sub-steps (AgreementOnGenerators branches)

  +-- generator branch: (d * d)(f) -> 0(f)
  |   (3 sub-step(s))
  |  [0] product-rule
  |       ((d * d)(f) + (-0(f)))  ->  (d(d(f)) + (-0))
  |  [1] d² = 0
  |       d(d(f))  ->  0
  |  [2] simplify
  |       (0 + (-0))  ->  0
  +--

  +-- generator branch: (d * d)(d(f)) -> 0(d(f))
  |   (4 sub-step(s))
  |  [0] product-rule
  |       ((d * d)(d(f)) + (-0(d(f))))  ->  (d(d(d(f))) + (-0))
  |  [1] d² = 0
  |       d(d(f))  ->  0
  |  [2] product-rule
  |       (d(0) + (-0))  ->  (0 + (-0))
  |  [3] simplify
  |       (0 + (-0))  ->  0
  +--


## 4. Relation (2), $d\mathcal{L}_X - \mathcal{L}_X d = 0$

**Operator equation:** $[d, \mathcal{L}_X] \equiv 0$, i.e. $d$ and
$\mathcal{L}_X$ commute.

**Per generator:**
* On $f$: $d\mathcal{L}_X(f) - \mathcal{L}_X(df) = d(X(f)) - d(X(f)) = 0$ (Cartan magic + $d^2 = 0$).
* On $df$: $d\mathcal{L}_X(df) - \mathcal{L}_X(d(df)) = d\mathcal{L}_X(df) - 0 = d\mathcal{L}_X(df)$;
  Cartan magic gives $\mathcal{L}_X(df) = d\iota_X(df) + \iota_X(d^2 f) = d(X(f))$;
  applying the outer $d$ yields $d^2(X(f)) = 0$.

In [5]:
L_X = lie(X)
chain2 = cartan.verify("d_lie", algebra=algebra, X=X, registry=reg)
print(f"verify(d_lie)  ->  CLOSED ({len(chain2)} steps, root rule: {chain2.steps[0].rule!r})")
print()
show_relation(
    "(2) [d, L_X] = 0",
    lhs_op=Sum(compose(d, L_X), -compose(L_X, d)),
    rhs_op=Integer(0),
)
show_branches(chain2, label="(2) [d, L_X] = 0")


verify(d_lie)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (2) [d, L_X] = 0 ---
  OK  generator f       LHS(f) = 0
                RHS(f) = 0
  OK  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (2) [d, L_X] = 0, sub-steps (AgreementOnGenerators branches)

  +-- generator branch: ((d * L_X) + (-(L_X * d)))(f) -> 0(f)
  |   (12 sub-step(s))
  |  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  |       ((d * L_X) + (-(L_X * d)))(f)  ->  ((d * L_X)(f) + (-(L_X * d))(f))
  |  [1] product-rule
  |       (((d * L_X)(f) + (-(L_X * d))(f)) + (-0(f)))  ->  ((d(L_X(f)) + (-L_X(d(f)))) + (-0))
  |  [2] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  |       L_X(f)  ->  ((d * ι_X)(f) + (ι_X * d)(f))
  |  [3] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  |       L_X(d(f))  ->  ((d * ι_X)(d(f)) + (ι_X * d)(d(f)))
  |  [4] product-rule
  |       ((d(((d * ι_X)(f) + (ι_X * d)(f))) + (-((d * ι_X)(d(f)) + (ι_X * d)(d(f))))) + (-0))  ->  (((d(d(ι_X(f))) + d(ι_X(d(f)))) + (-(d(ι

## 5. Relation (3), $d\iota_X + \iota_X d = \mathcal{L}_X$ (Cartan magic)

**Operator equation:** $\mathcal{L}_X \equiv d\iota_X + \iota_X d$.

In Cartan-mode the package defines $\mathcal{L}_X$ **exactly by this
formula**, i.e. relation (3) is the statement of the
`LieDerivativeCartanDefinition` axiom.

**Per generator:**
* On $f$: $d(\iota_X f) + \iota_X(df) = d(0) + X(f) = X(f) = \mathcal{L}_X(f)$.
* On $df$: $d(\iota_X(df)) + \iota_X(d^2 f) = d(X(f)) + 0 = d(X(f)) = \mathcal{L}_X(df)$.

In [6]:
iota_X = iota(X)
chain3 = cartan.verify("cartan_magic", algebra=algebra, X=X, registry=reg)
print(f"verify(cartan_magic)  ->  CLOSED ({len(chain3)} steps, root rule: {chain3.steps[0].rule!r})")
print()
show_relation(
    "(3) d ι_X + ι_X d = L_X",
    lhs_op=Sum(compose(d, iota_X), compose(iota_X, d)),
    rhs_op=L_X,
)
show_branches(chain3, label="(3) d ι_X + ι_X d = L_X")


verify(cartan_magic)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (3) d ι_X + ι_X d = L_X ---
  OK  generator f       LHS(f) = X(f)
                RHS(f) = X(f)
  OK  generator d(f)    LHS(d(f)) = d(X(f))
                RHS(d(f)) = d(X(f))

### (3) d ι_X + ι_X d = L_X, sub-steps (AgreementOnGenerators branches)

  +-- generator branch: ((d * ι_X) + (ι_X * d))(f) -> L_X(f)
  |   (9 sub-step(s))
  |  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  |       ((d * ι_X) + (ι_X * d))(f)  ->  ((d * ι_X)(f) + (ι_X * d)(f))
  |  [1] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  |       L_X(f)  ->  ((d * ι_X)(f) + (ι_X * d)(f))
  |  [2] product-rule
  |       (((d * ι_X)(f) + (ι_X * d)(f)) + (-((d * ι_X)(f) + (ι_X * d)(f))))  ->  ((d(ι_X(f)) + ι_X(d(f))) + (-(d(ι_X(f)) + ι_X(d(f)))))
  |  [3] ι_X(f) = 0 on 0-forms
  |       ι_X(f)  ->  0
  |  [4] ι_X(df) = X(f)
  |       ι_X(d(f))  ->  X(f)
  |  [5] ι_X(f) = 0 on 0-forms
  |       ι_X(f)  ->  0
  |  [6] ι_X(df) = X(f)
  |       ι_X

## 6. Relation (4), $\mathcal{L}_X\mathcal{L}_Y - \mathcal{L}_Y\mathcal{L}_X = \mathcal{L}_{[X,Y]}$

**Operator equation:** $[\mathcal{L}_X, \mathcal{L}_Y] \equiv \mathcal{L}_{[X,Y]}$.

The vector-field bracket $[X, Y]$ is the composition $X\cdot Y - Y\cdot X$
implemented by `LieBracket`.

**Per generator:**
* On $f$: $\mathcal{L}_X\mathcal{L}_Y(f) - \mathcal{L}_Y\mathcal{L}_X(f) = X(Y(f)) - Y(X(f)) = [X,Y](f) = \mathcal{L}_{[X,Y]}(f)$.
* On $df$: open Cartan magic on both sides and simplify with $d^2 = 0$ + $\iota_X(df) = X(f)$.

In [7]:
L_Y = lie(Y)
XY = LieBracket().expand(X, Y)
L_XY = lie(XY)
chain4 = cartan.verify("lie_lie", algebra=algebra, X=X, Y=Y, registry=reg)
print(f"verify(lie_lie)  ->  CLOSED ({len(chain4)} steps, root rule: {chain4.steps[0].rule!r})")
print()
show_relation(
    "(4) [L_X, L_Y] = L_[X,Y]",
    lhs_op=Sum(compose(L_X, L_Y), -compose(L_Y, L_X)),
    rhs_op=L_XY,
)
show_branches(chain4, label="(4) [L_X, L_Y] = L_[X,Y]")


verify(lie_lie)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (4) [L_X, L_Y] = L_[X,Y] ---
  OK  generator f       LHS(f) = (X(Y(f)) + (-Y(X(f))))
                RHS(f) = (X(Y(f)) + (-Y(X(f))))
  OK  generator d(f)    LHS(d(f)) = (d(X(Y(f))) + (-d(Y(X(f)))))
                RHS(d(f)) = (d(X(Y(f))) + (-d(Y(X(f)))))

### (4) [L_X, L_Y] = L_[X,Y], sub-steps (AgreementOnGenerators branches)

  +-- generator branch: ((L_X * L_Y) + (-(L_Y * L_X)))(f) -> L_((X * Y) + (-(Y * X)))(f)
  |   (31 sub-step(s))
  |  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  |       ((L_X * L_Y) + (-(L_Y * L_X)))(f)  ->  ((L_X * L_Y)(f) + (-(L_Y * L_X))(f))
  |  [1] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  |       L_((X * Y) + (-(Y * X)))(f)  ->  ((d * ι_((X * Y) + (-(Y * X))))(f) + (ι_((X * Y) + (-(Y * X))) * d)(f))
  |  [2] product-rule
  |       (((L_X * L_Y)(f) + (-(L_Y * L_X))(f)) + (-((d * ι_((X * Y) + (-(Y * X))))(f) + (ι_((X * Y) + (-(Y * X))) * d)(f))))  ->  ((L_X(L_Y(f)) + (-L_Y(L_X(

## 7. Relation (5), $\mathcal{L}_X\iota_Y - \iota_Y\mathcal{L}_X = \iota_{[X,Y]}$

**Operator equation:** $[\mathcal{L}_X, \iota_Y] \equiv \iota_{[X,Y]}$.

**Per generator:**
* On $f$: $\mathcal{L}_X(\iota_Y f) - \iota_Y(\mathcal{L}_X f) = \mathcal{L}_X(0) - \iota_Y(X(f)) = 0 - 0 = 0$;
  also $\iota_{[X,Y]}(f) = 0$ (interior is zero on 0-forms).
* On $df$: $\mathcal{L}_X(\iota_Y(df)) - \iota_Y(\mathcal{L}_X(df)) = \mathcal{L}_X(Y(f)) - \iota_Y(d(X(f))) = X(Y(f)) - Y(X(f)) = [X,Y](f) = \iota_{[X,Y]}(df)$.

In [8]:
iota_Y = iota(Y)
iota_XY = iota(XY)
chain5 = cartan.verify("lie_iota", algebra=algebra, X=X, Y=Y, registry=reg)
print(f"verify(lie_iota)  ->  CLOSED ({len(chain5)} steps, root rule: {chain5.steps[0].rule!r})")
print()
show_relation(
    "(5) [L_X, ι_Y] = ι_[X,Y]",
    lhs_op=Sum(compose(L_X, iota_Y), -compose(iota_Y, L_X)),
    rhs_op=iota_XY,
)
show_branches(chain5, label="(5) [L_X, ι_Y] = ι_[X,Y]")


verify(lie_iota)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (5) [L_X, ι_Y] = ι_[X,Y] ---
  OK  generator f       LHS(f) = 0
                RHS(f) = 0
  OK  generator d(f)    LHS(d(f)) = (X(Y(f)) + (-Y(X(f))))
                RHS(d(f)) = (X(Y(f)) + (-Y(X(f))))

### (5) [L_X, ι_Y] = ι_[X,Y], sub-steps (AgreementOnGenerators branches)

  +-- generator branch: ((L_X * ι_Y) + (-(ι_Y * L_X)))(f) -> ι_((X * Y) + (-(Y * X)))(f)
  |   (13 sub-step(s))
  |  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  |       ((L_X * ι_Y) + (-(ι_Y * L_X)))(f)  ->  ((L_X * ι_Y)(f) + (-(ι_Y * L_X))(f))
  |  [1] ι_X(f) = 0 on 0-forms
  |       ι_((X * Y) + (-(Y * X)))(f)  ->  0
  |  [2] product-rule
  |       (((L_X * ι_Y)(f) + (-(ι_Y * L_X))(f)) + (-0))  ->  ((L_X(ι_Y(f)) + (-ι_Y(L_X(f)))) + (-0))
  |  [3] ι_X(f) = 0 on 0-forms
  |       ι_Y(f)  ->  0
  |  [4] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  |       L_X(0)  ->  ((d * ι_X)(0) + (ι_X * d)(0))
  |  [5] L_X := d∘ι_X + ι_X∘d (Cartan def

## 8. Relation (6), $\iota_X\iota_Y + \iota_Y\iota_X = 0$

**Operator equation:** Interior product is graded skew (degree $-1$,
so two interiors anti-commute).

This relation is not in `CartanCalculus.RELATIONS`, we prove it
directly with `prove_operator_equation` using `AgreementOnGenerators`.

**Per generator:**
* On $f$: $\iota_X(\iota_Y f) + \iota_Y(\iota_X f) = \iota_X(0) + \iota_Y(0) = 0$ (interior is zero on 0-forms).
* On $df$: $\iota_X(\iota_Y(df)) + \iota_Y(\iota_X(df)) = \iota_X(Y(f)) + \iota_Y(X(f)) = 0 + 0 = 0$ ($Y(f), X(f)$ are 0-forms).

So both sides reduce to $0$ on both generators.

In [9]:
lhs6 = Sum(compose(iota_X, iota_Y), compose(iota_Y, iota_X))
rhs6 = Integer(0)

chain6 = prove_operator_equation(
    lhs6, rhs6, algebra,
    registry=reg,
    engine=engine,
)
print(f"prove_operator_equation(ι_X∘ι_Y + ι_Y∘ι_X = 0)  ->  CLOSED ({len(chain6)} steps, root rule: {chain6.steps[0].rule!r})")
print()
show_relation("(6) ι_X∘ι_Y + ι_Y∘ι_X = 0", lhs_op=lhs6, rhs_op=rhs6)
show_branches(chain6, label="(6) ι_X∘ι_Y + ι_Y∘ι_X = 0")


prove_operator_equation(ι_X∘ι_Y + ι_Y∘ι_X = 0)  ->  CLOSED (1 steps, root rule: 'AgreementOnGenerators')

--- (6) ι_X∘ι_Y + ι_Y∘ι_X = 0 ---
  OK  generator f       LHS(f) = 0
                RHS(f) = 0
  OK  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (6) ι_X∘ι_Y + ι_Y∘ι_X = 0, sub-steps (AgreementOnGenerators branches)

  +-- generator branch: ((ι_X * ι_Y) + (ι_Y * ι_X))(f) -> 0(f)
  |   (7 sub-step(s))
  |  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  |       ((ι_X * ι_Y) + (ι_Y * ι_X))(f)  ->  ((ι_X * ι_Y)(f) + (ι_Y * ι_X)(f))
  |  [1] product-rule
  |       (((ι_X * ι_Y)(f) + (ι_Y * ι_X)(f)) + (-0(f)))  ->  ((ι_X(ι_Y(f)) + ι_Y(ι_X(f))) + (-0))
  |  [2] ι_X(f) = 0 on 0-forms
  |       ι_Y(f)  ->  0
  |  [3] ι_X(f) = 0 on 0-forms
  |       ι_X(0)  ->  0
  |  [4] ι_X(f) = 0 on 0-forms
  |       ι_X(f)  ->  0
  |  [5] ι_X(f) = 0 on 0-forms
  |       ι_Y(0)  ->  0
  |  [6] simplify
  |       ((0 + 0) + (-0))  ->  0
  +--

  +-- generator branch: ((ι_X * ι_Y) + (ι

## 9. Single-pass verification, `verify_all`

Relations (1)-(5) can be swept in a single call; (6) was handled
separately above.

In [10]:
results = cartan.verify_all(algebra=algebra, X=X, Y=Y, registry=reg)
print("verify_all results:")
for name, chain in results.items():
    parent = chain.steps[0]
    n_children = len(parent.children) if parent.children else 0
    print(f"  {name:<18}  {len(chain)} steps, {n_children} sub-generator branches, root: {parent.rule!r}")
print(f"\nRelation (6) separately: {len(chain6)} steps, root: {chain6.steps[0].rule!r}")
print(f"\n6/6 relations closed.")

verify_all results:
  d_squared_zero      1 steps, 2 sub-generator branches, root: 'AgreementOnGenerators'
  cartan_magic        1 steps, 2 sub-generator branches, root: 'AgreementOnGenerators'
  d_lie               1 steps, 2 sub-generator branches, root: 'AgreementOnGenerators'
  lie_lie             1 steps, 2 sub-generator branches, root: 'AgreementOnGenerators'
  lie_iota            1 steps, 2 sub-generator branches, root: 'AgreementOnGenerators'

Relation (6) separately: 1 steps, root: 'AgreementOnGenerators'

6/6 relations closed.


## 10. Inspecting one step in LaTeX, Cartan magic ($f$ branch)

`AgreementOnGenerators` produces a sub-chain per generator. Let us
render the $f$-branch of relation (3).

In [11]:
from jacopy.display.jupyter import display_step

# Bonus: render the Cartan magic (3) f-branch in LaTeX.
parent = chain3.steps[0]
f_branch = parent.children[0]
print(f"LaTeX render, Cartan magic, f branch:")
display_step(f_branch)


LaTeX render, Cartan magic, f branch:


\begin{gather*}
\left(d \, \iota_X + \iota_X \, d\right)\!\left(f\right) \to L_X\!\left(f\right) \quad \text{[check-on-generator]}\;\text{--- generator g = f}
\end{gather*}

## Conclusion

$$
\boxed{\text{The six (1)-(6) close on the generator pair } \{f, df\}\text{ of } \Omega^\bullet(M)\text{ via } \texttt{AgreementOnGenerators}.}
$$

Key points:

* The generator set of `ExteriorAlgebra((f,))` is $\{f, df\}$, i.e.
  **one 0-form and one 1-form**. Each relation is proved separately on
  these two generators; `AgreementOnGenerators` lifts equality on the
  generator set to equality on the whole algebra (for graded
  derivations).
* (3) Cartan magic holds *by definition*, in the package, cartan-mode
  $\mathcal{L}_X$ is set up exactly as $d\iota_X + \iota_X d$.
* (1), (4), (6) build on the axiom set in `default_engine`:
  $d^2 = 0$, $\iota_X^2 = 0$, $\iota_X(f) = 0$, $\iota_X(df) = X(f)$.
* (2), (5) follow from combining two definitions: the Cartan
  definition of $\mathcal{L}_X$ + $d^2 = 0$ + $\iota_X$-action rules.
* (6) is not in the `RELATIONS` list, so we appealed directly to
  `prove_operator_equation` with the same strategy.

This notebook documents the operator-level closure of Cartan calculus
**explicitly**, the output for each generator (0-form, 1-form) is
marked with `OK`.